In [47]:
import pandas as pd


In [48]:
df = pd.read_csv('C:\\Users\\91947\Desktop\\round 2 flipkart\\Flipkart-round-2-\\data\\Cleaned_Dataset.csv')
df.head()
df.info()

<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:1: SyntaxWarning: invalid escape sequence '\D'
C:\Users\91947\AppData\Local\Temp\ipykernel_30412\2681524173.py:1: SyntaxWarning: invalid escape sequence '\D'
  df = pd.read_csv('C:\\Users\\91947\Desktop\\round 2 flipkart\\Flipkart-round-2-\\data\\Cleaned_Dataset.csv')


<class 'pandas.DataFrame'>
RangeIndex: 8168 entries, 0 to 8167
Data columns (total 36 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Unnamed: 0             8168 non-null   int64  
 1   id                     8168 non-null   str    
 2   event_type             8168 non-null   str    
 3   latitude               8168 non-null   float64
 4   longitude              8168 non-null   float64
 5   endlatitude            8168 non-null   float64
 6   endlongitude           8168 non-null   float64
 7   address                8168 non-null   str    
 8   end_address            8168 non-null   str    
 9   event_cause            8168 non-null   str    
 10  requires_road_closure  8168 non-null   bool   
 11  start_datetime         8168 non-null   str    
 12  end_datetime           8168 non-null   str    
 13  status                 8168 non-null   str    
 14  authenticated          8168 non-null   str    
 15  modified_dateti

## Remove unnecessary columns 

In [49]:
drop_cols = [
    'id',
    'client_id',
    'created_by_id',
    'last_modified_by_id',
    'authenticated',
    'description',
]
df.drop(columns=drop_cols, inplace=True)


Changing the data column into proper format for better analysis

In [50]:
datetime_cols = [
    'start_datetime',
    'end_datetime',
    'closed_datetime',
    'modified_datetime',
    'created_date'
]

for col in datetime_cols:
    df[col] = pd.to_datetime(
        df[col],
        errors='coerce',
        format='mixed'
    )

How long did the disruption last?

In [51]:
df['event_duration_min'] = (
    df['end_datetime'] -
    df['start_datetime']
).dt.total_seconds()/60


Closure Delay

In [52]:
df['closure_delay_min'] = (
    df['closed_datetime'] -
    df['end_datetime']
).dt.total_seconds()/60

Hour

In [53]:
df['hour'] = df['start_datetime'].dt.hour

Day of Week

In [54]:
df['day_of_week'] = df['start_datetime'].dt.dayofweek

Weekend Flag

In [55]:
df['is_weekend'] = (
    df['day_of_week'] >= 5
).astype(int)


Month

In [56]:
df['month'] = df['start_datetime'].dt.month


Peak Hour

In [57]:
df['peak_hour'] = (
    df['hour'].between(7,10) |
    df['hour'].between(16,20)
).astype(int)

here we are seeing the area affected in the event

In [58]:
from geopy.distance import geodesic

def distance_km(row):
    return geodesic(
        (row['latitude'], row['longitude']),
        (row['endlatitude'], row['endlongitude'])
    ).km

df['stretch_length_km'] = df.apply(
    distance_km,
    axis=1
)

here we measure wheter the event is moving or not 

In [59]:
def resolved_distance(row):
    return geodesic(
        (row['latitude'], row['longitude']),
        (
            row['resolved_at_latitude'],
            row['resolved_at_longitude']
        )
    ).km

df['resolution_shift_km'] = df.apply(
    resolved_distance,
    axis=1
)

How often does this corridor experience events?

In [60]:
corridor_freq = (
    df['corridor']
    .value_counts()
)
df['corridor_event_count'] = (
    df['corridor']
    .map(corridor_freq)
)

Corridor Risk Score

In [61]:
closure_rate = (
    df.groupby('corridor')
      ['requires_road_closure']
      .mean()
)

Junction Event Count

In [62]:
junction_freq = (
    df['junction']
      .value_counts()
)
df['junction_event_count'] = (
    df['junction']
      .map(junction_freq)
)

Junction Priority Rate

In [63]:
priority_rate = (
    df.groupby('junction')
      ['priority']
      .apply(
          lambda x:(x=='High').mean()
      )
)

Vehicle Severity Score

In [64]:
severity_map = {
    'two_wheeler':1,
    'auto':2,
    'private_car':2,
    'lcv':3,
    'bmtc_bus':4,
    'heavy_vehicle':5,
    'others':1
}

df['vehicle_severity'] = (
    df['veh_type']
      .map(severity_map)
)

Cargo Presence

In [65]:
df['has_cargo'] = (
    df['cargo_material']
    != 'no goods'
).astype(int)

Create Cause Severity Score

In [66]:
def cause_group(cause):

    cause = str(cause).lower()

    if 'accident' in cause:
        return 'Accident'

    elif 'breakdown' in cause:
        return 'Breakdown'

    elif 'water' in cause or 'flood' in cause:
        return 'Flooding'

    elif 'signal' in cause:
        return 'Signal Failure'

    elif 'pothole' in cause or 'road' in cause:
        return 'Road Damage'

    else:
        return 'Other'

df['cause_group'] = df['event_cause'].apply(cause_group)
cause_score = {
    'Road Damage':1,
    'Signal Failure':2,
    'Breakdown':3,
    'Accident':4,
    'Flooding':5,
    'Other':2
}

df['cause_severity'] = (
    df['cause_group']
      .map(cause_score)
)

In [67]:
df['diversion_required'] = (
    (df['requires_road_closure'] == True) |
    (df['priority'] == 'High')
).astype(int)

In [68]:
print(df['priority'].value_counts())

priority
High    5027
Low     3141
Name: count, dtype: int64


In [69]:
def manpower_category(row):

    score = 0

    # Priority
    if row['priority'] == 'High':
        score += 3
    else:
        score += 1

    # Road Closure
    if row['requires_road_closure']:
        score += 2

    # Vehicle Severity
    if row['vehicle_severity'] >= 4:
        score += 2

    # Peak Hour
    if row['peak_hour'] == 1:
        score += 1

    # Long Duration
    if row['event_duration_min'] > 60:
        score += 1

    return score

df['manpower_score'] = df.apply(
    manpower_category,
    axis=1
)

In [70]:
df['manpower_score'].value_counts().sort_index()

manpower_score
1     972
2     725
3    2427
4    1838
5    1324
6     715
7     137
8      26
9       4
Name: count, dtype: int64

In [71]:
def manpower_label(score):

    if score <= 3:
        return 'Low'

    elif score <= 5:
        return 'Medium'

    else:
        return 'High'

df['manpower_required'] = (
    df['manpower_score']
      .apply(manpower_label)
)
df['manpower_required'].value_counts()

manpower_required
Low       4124
Medium    3162
High       882
Name: count, dtype: int64

In [72]:
df['requires_road_closure'].value_counts()

requires_road_closure
False    7494
True      674
Name: count, dtype: int64

In [73]:
df['manpower_required'].value_counts()

manpower_required
Low       4124
Medium    3162
High       882
Name: count, dtype: int64

In [74]:
df['priority'].value_counts()

priority
High    5027
Low     3141
Name: count, dtype: int64

## Encode Categorical Variables

In [75]:
from sklearn.preprocessing import LabelEncoder

# Targets
target_cols = [
    'priority',
    'requires_road_closure',
    'diversion_required',
    'manpower_required'
]

# Features
X = df.drop(columns=target_cols)

# Encode categorical columns once
encoders = {}

for col in X.select_dtypes(include='object').columns:

    le = LabelEncoder()

    X[col] = le.fit_transform(
        X[col].astype(str)
    )

    encoders[col] = le

print("Feature matrix shape:", X.shape)

Feature matrix shape: (8168, 44)


C:\Users\91947\AppData\Local\Temp\ipykernel_30412\787283439.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X.select_dtypes(include='object').columns:


Removed duplicate values 

In [76]:
df = df.drop_duplicates(
    subset=[
        'event_type',
        'event_cause',
        'address',
        'start_datetime'
    ]
)

In [77]:
print(X.select_dtypes(include='object').columns.tolist())

[]


In [78]:
print(
    X.select_dtypes(
        include=['object', 'string']
    ).columns.tolist()
)

[]


verify feature importance

In [79]:
df.duplicated().sum()

np.int64(0)

In [80]:
df.duplicated(subset=[
    'event_type',
    'event_cause',
    'address',
    'start_datetime'
]).sum()

np.int64(0)

## Create early warning features

In [81]:
early_features = [

    # Incident Information
    'event_type',
    'event_cause',
    'veh_type',

    # Location
    'latitude',
    'longitude',

    # Time Features
    'hour',
    'peak_hour',
    'month',
    'is_weekend',

    # Engineered Severity Features
    'vehicle_severity',
    'cause_severity'
]

In [82]:
X_early = df[early_features].copy()

## Label encoding 

In [108]:
from sklearn.preprocessing import LabelEncoder

early_encoders = {}

for col in X_early.select_dtypes(
    include=['object', 'string']
).columns:

    le = LabelEncoder()

    X_early[col] = le.fit_transform(
        X_early[col].astype(str)
    )

    early_encoders[col] = le

print("Early Warning Encoding Complete")

Early Warning Encoding Complete


In [84]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

def train_model(X, y, model_name):

    print("\n" + "="*60)
    print(f"MODEL: {model_name}")
    print("="*60)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    model = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight='balanced'
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    print("\nAccuracy:")
    print(accuracy_score(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    return model

In [85]:
y_priority = df['priority']

In [86]:
priority_model = train_model(
    X_early,
    y_priority,
    "Priority Early Warning"
)


MODEL: Priority Early Warning

Accuracy:
0.8855905998763142

Classification Report:
              precision    recall  f1-score   support

        High       0.87      0.95      0.91      1000
         Low       0.91      0.77      0.84       617

    accuracy                           0.89      1617
   macro avg       0.89      0.86      0.87      1617
weighted avg       0.89      0.89      0.88      1617


Confusion Matrix:
[[955  45]
 [140 477]]


In [87]:
y_barricade = (
    df['requires_road_closure']
    .astype(int)
)

In [88]:
barricade_model = train_model(
    X_early,
    y_barricade,
    "Barricade Early Warning"
)


MODEL: Barricade Early Warning

Accuracy:
0.9313543599257885

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.99      0.96      1484
           1       0.74      0.26      0.38       133

    accuracy                           0.93      1617
   macro avg       0.84      0.62      0.67      1617
weighted avg       0.92      0.93      0.92      1617


Confusion Matrix:
[[1472   12]
 [  99   34]]


In [89]:
y_diversion = df['diversion_required']

In [90]:
diversion_model = train_model(
    X_early,
    y_diversion,
    "Diversion Early Warning"
)


MODEL: Diversion Early Warning

Accuracy:
0.862708719851577

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.69      0.77       542
           1       0.86      0.95      0.90      1075

    accuracy                           0.86      1617
   macro avg       0.87      0.82      0.84      1617
weighted avg       0.86      0.86      0.86      1617


Confusion Matrix:
[[ 374  168]
 [  54 1021]]


In [91]:
y_manpower = df['manpower_required']

In [92]:
manpower_model = train_model(
    X_early,
    y_manpower,
    "Manpower Early Warning"
)


MODEL: Manpower Early Warning

Accuracy:
0.7835497835497836

Classification Report:
              precision    recall  f1-score   support

        High       0.75      0.62      0.68       176
         Low       0.84      0.86      0.85       813
      Medium       0.72      0.73      0.73       628

    accuracy                           0.78      1617
   macro avg       0.77      0.74      0.75      1617
weighted avg       0.78      0.78      0.78      1617


Confusion Matrix:
[[110   6  60]
 [  0 697 116]
 [ 37 131 460]]


Feature Importance Function

In [93]:
import pandas as pd

def show_feature_importance(model, X):

    importance = pd.DataFrame({

        'Feature': X.columns,

        'Importance':
        model.feature_importances_

    })

    importance = importance.sort_values(
        by='Importance',
        ascending=False
    )

    return importance

Priority importance

In [94]:
show_feature_importance(
    priority_model,
    X_early
).head(10)

,Feature,Importance
4,longitude,0.396851
3,latitude,0.320000
5,hour,0.088340
7,month,0.060217
1,event_cause,0.036956
2,veh_type,0.027032
8,is_weekend,0.019630
9,vehicle_severity,0.019622
10,cause_severity,0.014184
6,peak_hour,0.012043


Barricade importance 

In [95]:
show_feature_importance(
    barricade_model,
    X_early
).head(10)

,Feature,Importance
4,longitude,0.239354
3,latitude,0.238493
5,hour,0.119207
1,event_cause,0.094982
7,month,0.075363
10,cause_severity,0.072720
9,vehicle_severity,0.044245
2,veh_type,0.038658
0,event_type,0.037265
8,is_weekend,0.022512


Diversion Importance 

In [96]:
show_feature_importance(
    diversion_model,
    X_early
).head(10)

,Feature,Importance
4,longitude,0.381051
3,latitude,0.320262
5,hour,0.090767
7,month,0.062545
1,event_cause,0.037052
2,veh_type,0.029275
9,vehicle_severity,0.021134
8,is_weekend,0.020633
10,cause_severity,0.017431
6,peak_hour,0.012031


ManPower Importance 

In [97]:
show_feature_importance(
    manpower_model,
    X_early
).head(10)

,Feature,Importance
4,longitude,0.224119
3,latitude,0.205182
5,hour,0.118444
9,vehicle_severity,0.104286
6,peak_hour,0.103116
2,veh_type,0.097182
7,month,0.061024
1,event_cause,0.036808
8,is_weekend,0.019457
10,cause_severity,0.018728


## Operational Models 
basically in this model we get more info about the event to take more precise decision

In [98]:
operational_features = [

    'event_type',
    'event_cause',
    'veh_type',

    'corridor',
    'police_station',
    'zone',
    'junction',

    'latitude',
    'longitude',

    'hour',
    'peak_hour',
    'month',
    'is_weekend',

    'vehicle_severity',
    'cause_severity',

    'stretch_length_km',

    'corridor_event_count',
    'junction_event_count'
]

In [99]:
X_operational = df[operational_features].copy()

In [112]:
from sklearn.preprocessing import LabelEncoder

operational_encoders = {}

for col in X_operational.select_dtypes(
    include=['object', 'string']
).columns:

    le = LabelEncoder()

    X_operational[col] = le.fit_transform(
        X_operational[col].astype(str)
    )

    operational_encoders[col] = le

print("Operational Encoding Complete")

Operational Encoding Complete


In [101]:
y_priority = df['priority']

priority_operational_model = train_model(
    X_operational,
    y_priority,
    "Operational Priority Model"
)


MODEL: Operational Priority Model

Accuracy:
1.0

Classification Report:
              precision    recall  f1-score   support

        High       1.00      1.00      1.00      1000
         Low       1.00      1.00      1.00       617

    accuracy                           1.00      1617
   macro avg       1.00      1.00      1.00      1617
weighted avg       1.00      1.00      1.00      1617


Confusion Matrix:
[[1000    0]
 [   0  617]]


In [102]:
y_barricade = (
    df['requires_road_closure']
    .astype(int)
)

barricade_operational_model = train_model(
    X_operational,
    y_barricade,
    "Operational Barricade Model"
)


MODEL: Operational Barricade Model

Accuracy:
0.9956709956709957

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1484
           1       0.96      0.98      0.97       133

    accuracy                           1.00      1617
   macro avg       0.98      0.99      0.99      1617
weighted avg       1.00      1.00      1.00      1617


Confusion Matrix:
[[1479    5]
 [   2  131]]


In [103]:
y_diversion = (
    df['diversion_required']
)

diversion_operational_model = train_model(
    X_operational,
    y_diversion,
    "Operational Diversion Model"
)


MODEL: Operational Diversion Model

Accuracy:
0.9962894248608535

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       542
           1       1.00      1.00      1.00      1075

    accuracy                           1.00      1617
   macro avg       1.00      1.00      1.00      1617
weighted avg       1.00      1.00      1.00      1617


Confusion Matrix:
[[ 539    3]
 [   3 1072]]


In [104]:
y_manpower = (
    df['manpower_required']
)

manpower_operational_model = train_model(
    X_operational,
    y_manpower,
    "Operational Manpower Model"
)


MODEL: Operational Manpower Model

Accuracy:
0.904143475572047

Classification Report:
              precision    recall  f1-score   support

        High       0.93      0.80      0.86       176
         Low       0.90      0.97      0.94       813
      Medium       0.90      0.85      0.87       628

    accuracy                           0.90      1617
   macro avg       0.91      0.87      0.89      1617
weighted avg       0.90      0.90      0.90      1617


Confusion Matrix:
[[140   0  36]
 [  0 787  26]
 [ 10  83 535]]


In [105]:
import joblib
import os

os.makedirs("models", exist_ok=True)

joblib.dump(
    priority_model,
    "models/priority_early.pkl"
)

joblib.dump(
    barricade_model,
    "models/barricade_early.pkl"
)

joblib.dump(
    diversion_model,
    "models/diversion_early.pkl"
)

joblib.dump(
    manpower_model,
    "models/manpower_early.pkl"
)

print("Early Warning Models Saved")

Early Warning Models Saved


In [107]:
import joblib
import os

os.makedirs("models", exist_ok=True)

joblib.dump(
    priority_operational_model,
    "models/priority_operational.pkl"
)

joblib.dump(
    barricade_operational_model,
    "models/barricade_operational.pkl"
)

joblib.dump(
    diversion_operational_model,
    "models/diversion_operational.pkl"
)

joblib.dump(
    manpower_operational_model,
    "models/manpower_operational.pkl"
)

print("Operational Models Saved")

Operational Models Saved


In [109]:
import os

os.makedirs("encoders", exist_ok=True)

In [110]:
import joblib

joblib.dump(
    early_encoders,
    "encoders/early_encoders.pkl"
)

print("Early Encoders Saved")

Early Encoders Saved


In [113]:
joblib.dump(
    operational_encoders,
    "encoders/operational_encoders.pkl"
)

print("Operational Encoders Saved")

Operational Encoders Saved
